# Grover II – SageMath 10.8

Grover-inspirierte Inhaltssuche mit Amplitudenverstärkung für Obsidian-Vaults.

**Funktionen:**
- Relevanz-Scores (Suchbegriff-Vorkommen, log-Normierung)
- Grover-Amplitudenverstärkung (Oracle + Diffusion)
- FFT-Verstärkung im Frequenzraum
- Obsidian-Links, Datei-Öffnen

In [ ]:
from pathlib import Path
import numpy as np
from urllib.parse import quote
import os
import subprocess

# SageMath/Kernelnutzung
try:
    from sage.all import *
    print("✓ SageMath geladen")
except ImportError:
    print("Hinweis: SageMath nicht gefunden, nutze reines Python")

DEFAULT_VAULT = Path.home() / "Library/Mobile Documents/iCloud~md~obsidian/Documents/Zettel"

In [ ]:
def find_md_files(vault_path):
    """Sammelt alle .md-Dateien im Vault."""
    if not vault_path.exists():
        return []
    files = []
    for root, _, fnames in os.walk(vault_path):
        for f in fnames:
            if f.endswith('.md') and not f.startswith('.'):
                files.append(Path(root) / f)
    return files

def read_file(p):
    try:
        return p.read_text(encoding='utf-8', errors='replace')
    except:
        return ""

def url_encode(s):
    return quote(s, safe='')

def open_file(p, obsidian_url=None):
    """Öffnet Datei in Obsidian (macOS)."""
    try:
        if obsidian_url and os.name == 'posix':
            subprocess.run(['open', obsidian_url], check=False, capture_output=True)
            return
        path = str(p)
        if os.name == 'posix':
            subprocess.run(['open', '-a', 'Obsidian', path], check=False, capture_output=True)
        elif os.name == 'nt':
            os.startfile(path)
        else:
            subprocess.run(['xdg-open', path], check=False, capture_output=True)
    except Exception:
        pass

In [ ]:
class GroverSearch:
    """Grover-inspirierte Inhaltssuche mit Amplitudenverstärkung."""
    
    def __init__(self, file_paths, vault_root=None):
        self.files = list(file_paths)
        self.vault_root = vault_root or Path('.')
        self.content_cache = {}
    
    def _read_content(self, idx):
        if idx not in self.content_cache:
            self.content_cache[idx] = read_file(self.files[idx])
        return self.content_cache[idx]
    
    def relevance_scores(self, term, case_sensitive=False):
        term_l = term if case_sensitive else term.lower()
        scores = np.zeros(len(self.files), dtype=float)
        for i in range(len(self.files)):
            text = self._read_content(i)
            check = text if case_sensitive else text.lower()
            cnt = check.count(term_l)
            scores[i] = np.log1p(cnt) if cnt > 0 else 0.0
        mx = np.max(scores)
        if mx > 0:
            scores /= mx
        return scores
    
    def grover_amplify(self, scores, iterations):
        n = len(scores)
        if n == 0 or iterations < 1:
            return scores
        amp = np.where(scores > 0, -scores, scores).astype(float)
        for _ in range(iterations - 1):
            mean = np.mean(amp)
            amp = 2 * mean - amp
        mean = np.mean(amp)
        return np.abs(2 * mean - amp)
    
    def fft_amplify(self, scores):
        n = len(scores)
        if n == 0:
            return scores
        N = 1
        while N < n:
            N *= 2
        spec = np.fft.fft(np.pad(scores, (0, N - n)))
        mx = np.max(np.abs(spec)) + 1e-12
        spec *= (1 + 0.5 * np.abs(spec) / mx)
        out = np.fft.ifft(spec).real[:n]
        return np.abs(out)
    
    def search(self, term, case_sensitive=False, top_k=20, grover_iter=2, use_fft=True):
        scores = self.relevance_scores(term, case_sensitive)
        has_term = scores > 0
        if grover_iter > 0:
            scores = self.grover_amplify(scores, grover_iter)
        if use_fft:
            scores = self.fft_amplify(scores)
        order = np.argsort(scores)[::-1]
        results = []
        for k in order:
            if len(results) >= top_k:
                break
            if not has_term[k]:
                continue
            results.append((int(k), float(scores[k])))
        return results

## Interaktive Suche

Vault-Pfad und Suchbegriff unten anpassen.

In [ ]:
# === Parameter ===
VAULT = DEFAULT_VAULT.expanduser()
TERM = "St. Getreu"
TOP_K = 10
QUBITS = 0          # 0 = Standard (2 Iter.), sonst: ~ π/4 * 2^(Q/2)
USE_FFT = True
CASE_SENSITIVE = False

In [ ]:
if not VAULT.exists():
    print(f"Vault nicht gefunden: {VAULT}")
else:
    files = find_md_files(VAULT)
    print(f"Grover II – SageMath ({len(files)} Notizen)")
    
    grover_iter = 2
    if QUBITS > 0:
        grover_iter = max(1, int(3.14159/4 * 2**(QUBITS/2)))
    equiv_q = max(1, int(2 * np.log2(grover_iter * 4/3.14159 + 0.5))) if grover_iter > 0 else 0
    print(f"Grover: {grover_iter} Iterationen (~{equiv_q} Qubits)")
    
    gs = GroverSearch(files, VAULT)
    results = gs.search(TERM, CASE_SENSITIVE, TOP_K, grover_iter, USE_FFT)
    
    vault_name = VAULT.name or VAULT.parent.name or "Vault"
    term_l = TERM if CASE_SENSITIVE else TERM.lower()
    
    if not results:
        print("Keine Treffer.")
    else:
        print(f"\n{len(results)} Treffer für \"{TERM}\":\n")
        for r, (idx, score) in enumerate(results, 1):
            fp = files[idx]
            try:
                rel = fp.relative_to(VAULT)
            except:
                rel = fp.name
            rel_str = str(rel)
            obs_url = f"obsidian://open?vault={url_encode(vault_name)}&file={url_encode(rel_str)}"
            
            content = gs._read_content(idx)
            lines = content.split('\n')
            matches = []
            for i, line in enumerate(lines, 1):
                chk = line if CASE_SENSITIVE else line.lower()
                if term_l in chk:
                    matches.append((i, (line[:150] + '...') if len(line) > 150 else line))
            
            print(f"[{r}] --- {rel} (Score: {score:.4f}) ---")
            print(f"  Obsidian: {obs_url}")
            print(f"  Datei:    file://{fp}")
            if matches:
                for ln, snip in matches[:5]:
                    s = (snip[:80] + '...') if len(snip) > 80 else snip
                    print(f"  Z{ln}: {s}")
            else:
                print("  (Zeilenkontext nicht ermittelt)")
            print()
        
        choice = input(f"Zahl 1–{len(results)} zum Öffnen, Enter = Ende: ").strip()
        if choice:
            try:
                c = int(choice)
                if 1 <= c <= len(results):
                    idx = results[c-1][0]
                    fp = files[idx]
                    try:
                        rel = fp.relative_to(VAULT)
                    except:
                        rel = fp.name
                    obs_url = f"obsidian://open?vault={url_encode(vault_name)}&file={url_encode(str(rel))}"
                    open_file(fp, obs_url)
                    print(f"→ Geöffnet: {fp.name}")
            except ValueError:
                pass

## Mit ipywidgets (optional)

Falls `ipywidgets` installiert ist, kann eine interaktive Oberfläche genutzt werden.

In [ ]:
try:
    import ipywidgets as w
    from IPython.display import display, clear_output, HTML
    
    term_in = w.Text(description='Suchbegriff:', value='St. Getreu', style={'description_width': '100px'})
    vault_in = w.Text(description='Vault:', value=str(DEFAULT_VAULT), style={'description_width': '100px'})
    top_in = w.IntSlider(value=10, min=1, max=50, description='Top-K:')
    qub_in = w.IntSlider(value=0, min=0, max=16, description='Qubits:')
    out = w.Output()
    
    def do_search(btn):
        with out:
            clear_output(wait=True)
            VAULT = Path(vault_in.value).expanduser()
            TERM = term_in.value.strip()
            if not TERM or not VAULT.exists():
                print('Suchbegriff eingeben und Vault prüfen.')
                return
            files = find_md_files(VAULT)
            grover_iter = max(1, int(3.14159/4 * 2**(qub_in.value/2))) if qub_in.value > 0 else 2
            gs = GroverSearch(files, VAULT)
            results = gs.search(TERM, False, top_in.value, grover_iter, True)
            vault_name = VAULT.name or 'Vault'
            term_l = TERM.lower()
            for r, (idx, score) in enumerate(results, 1):
                fp = files[idx]
                try: rel = str(fp.relative_to(VAULT))
                except: rel = fp.name
                obs_url = f"obsidian://open?vault={url_encode(vault_name)}&file={url_encode(rel)}"
                content = gs._read_content(idx)
                matches = [(i, l[:80]) for i, l in enumerate(content.split('\n'), 1) if term_l in l.lower()][:3]
                print(f"[{r}] {rel} (Score: {score:.4f})")
                print(f"    {obs_url}")
                for ln, s in matches: print(f"    Z{ln}: {s}")
                print()
    
    btn = w.Button(description='Suchen')
    btn.on_click(do_search)
    display(w.VBox([term_in, vault_in, top_in, qub_in, btn, out]))
except ImportError:
    print('ipywidgets nicht installiert – nutze die Zellen oben.')